# CySent Unsloth Training (Colab)
A minimal notebook to fine-tune a small model for CySent action prediction.

In [ ]:
!pip -q install unsloth transformers datasets trl peft accelerate bitsandbytes
# Optional: clone project
# !git clone https://github.com/<your-org>/CySent.git
# %cd CySent

In [ ]:
import os
import json
from pathlib import Path
from datasets import load_dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments

# 1. Load dataset (with fallback generation for Colab)
dataset_path = 'datasets/cysent_action_dataset.jsonl'
if not os.path.exists(dataset_path):
    print("Dataset not found. Generating minimal dataset...")
    os.makedirs('datasets', exist_ok=True)
    data = [
        {"instruction": "Choose valid CySent action", "input": "risk=0.72, attack=phishing_email, target=auth_server, compromised=1", "output": "rotate_credentials"},
        {"instruction": "Choose valid CySent action", "input": "risk=0.85, attack=ransomware_detection, target=finance, compromised=1", "output": "isolate_suspicious_host"},
        {"instruction": "Choose valid CySent action", "input": "risk=0.45, attack=suspicious_login, target=web_server, compromised=0", "output": "increase_monitoring"},
        {"instruction": "Choose valid CySent action", "input": "risk=0.92, attack=lateral_movement, target=hospital, compromised=1", "output": "segment_finance_database"},
        {"instruction": "Choose valid CySent action", "input": "risk=0.15, attack=none, target=none, compromised=0", "output": "do_nothing"},
    ]
    with open(dataset_path, 'w') as f:
        for row in data:
            f.write(json.dumps(row) + '\n')
    print(f"Generated {len(data)} training samples.")

# 2. Load and split dataset
ds = load_dataset('json', data_files=dataset_path, split='train')
split_ds = ds.train_test_split(test_size=0.2, seed=42)
train_ds = split_ds['train']
eval_ds = split_ds['test']

# 3. Define valid actions and format
VALID_ACTIONS = [
    'do_nothing', 'patch_hr_systems', 'patch_web_server', 'patch_auth_server',
    'rotate_credentials', 'isolate_suspicious_host', 'increase_monitoring', 'restore_backup',
    'deploy_honeypot', 'phishing_training', 'investigate_top_alert', 'segment_finance_database'
]
ACTION_SET = ', '.join(VALID_ACTIONS)

# 4. Format dataset as text
def format_chat(example):
    prompt = (
        f"You are CySent BLUE defender. Choose exactly one action from: {ACTION_SET}\n"
        f"Instruction: {example['instruction']}\n"
        f"State: {example['input']}\n"
        f"Action:"
    )
    response = f" {example['output']}"
    return {'text': prompt + response}

train_data = train_ds.map(format_chat, remove_columns=train_ds.column_names)
eval_data = eval_ds.map(format_chat, remove_columns=eval_ds.column_names)

# 5. Load model with Unsloth (4-bit quantization)
model_name = 'Qwen/Qwen2.5-3B-Instruct'
max_seq = 512

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq,
    dtype=None,
    load_in_4bit=True,
)

# 6. Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

# 7. Create trainer
training_args = TrainingArguments(
    output_dir="outputs/cysent_checkpoint",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=5,
    num_train_epochs=1,
    learning_rate=2e-4,
    logging_steps=5,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_data,
    eval_dataset=eval_data,
    args=training_args,
    max_seq_length=max_seq,
    packing=False,
)

# 8. Train
print("Starting training...")
trainer.train()

# 9. Save adapter
os.makedirs("outputs/cysent_unsloth_adapter", exist_ok=True)
trainer.model.save_pretrained("outputs/cysent_unsloth_adapter")
tokenizer.save_pretrained("outputs/cysent_unsloth_adapter")
print("✓ Adapter saved to outputs/cysent_unsloth_adapter")

In [ ]:
import torch

# 1. Define action parser
def parse_action(text):
    lower = text.lower().strip()
    for action in VALID_ACTIONS:
        if action in lower:
            return action
    # Fallback to first word if it matches any action
    first_word = lower.split()[0] if lower.split() else ""
    for action in VALID_ACTIONS:
        if first_word == action:
            return action
    return 'do_nothing'

# 2. Test inference on trained model
FastLanguageModel.for_inference(model)  # Set to inference mode
test_state = "risk=0.72, attack=phishing_email, target=auth_server, compromised=1, credential_exposure=0.81"
test_prompt = (
    f"You are CySent BLUE defender. Choose exactly one action from: {ACTION_SET}\n"
    f"Instruction: Choose valid CySent action\n"
    f"State: {test_state}\n"
    f"Action:"
)

inputs = tokenizer(test_prompt, return_tensors="pt", truncation=True, max_length=512).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=20,
        temperature=0.0,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

raw_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
predicted_action = parse_action(raw_output.split("Action:")[-1])

print("=" * 60)
print("INFERENCE TEST")
print("=" * 60)
print(f"Test State: {test_state}")
print(f"Model Output: {raw_output.split('Action:')[-1].strip()}")
print(f"Parsed Action: {predicted_action}")
print("=" * 60)

In [ ]:
# 4. Validate saved adapter can be reloaded
print("\n✓ Training and inference complete!")
print("✓ Adapter saved. Validating reload...")

try:
    # Reload base model fresh with saved adapter
    model_reload, tokenizer_reload = FastLanguageModel.from_pretrained(
        model_name=model_name,
        max_seq_length=max_seq,
        dtype=None,
        load_in_4bit=True,
    )
    from peft import PeftModel
    model_reload = PeftModel.from_pretrained(model_reload, "outputs/cysent_unsloth_adapter")
    FastLanguageModel.for_inference(model_reload)
    print("✓ Adapter reload successful!")
    print("✓ Ready for production use or HF Hub push.")
except Exception as e:
    print(f"⚠ Reload check failed: {e}")
    print("  But adapter files were saved. Check manually if needed.")